# Notebook 02 — Full-Corpus Baseline Evaluation

**Goal:** Establish a dense retrieval baseline over the full ToolBench API corpus.

We build a FAISS inner-product index over all ~50k API embeddings and retrieve the top-k most similar APIs for each evaluation query. This measures how well `text-embedding-3-small` performs **without any fine-tuning** — the performance ceiling we aim to improve with contrastive training.

**Prerequisite:** Run `01_index_apis.ipynb` first to generate `corpus_embeddings.npy` and `api_names.json`.

## Setup

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / '.git').exists())
PROJECT_DIR = REPO_ROOT / 'project'
sys.path.insert(0, str(PROJECT_DIR))

load_dotenv(REPO_ROOT / '.env')

TOOLBENCH_DIR = Path(os.environ.get('TOOLBENCH_DIR', str(REPO_ROOT / 'toolbench_data')))
print(f"Repo:      {REPO_ROOT}")
print(f"Toolbench: {TOOLBENCH_DIR}")

## Load Data

Load the pre-computed API embeddings, API name index, and evaluation examples (747 queries with ground-truth API labels).

In [ ]:
import numpy as np, json
from data.load_toolbench import load_api_corpus, load_eval_examples
from models.embeddings import get_embeddings
from retrieval.retriever import build_faiss_index, retrieve_top_k
from evaluation.metrics import evaluate_batch

corpus = load_api_corpus(TOOLBENCH_DIR / "toolenv" / "tools")
corpus_matrix = np.load(PROJECT_DIR / "corpus_embeddings.npy")
with open(PROJECT_DIR / "api_names.json") as f:
    api_names = json.load(f)
name_to_idx = {name: i for i, name in enumerate(api_names)}
evals = load_eval_examples(TOOLBENCH_DIR / "toolllama_G123_dfs_eval.json")
print(f"Corpus: {len(corpus)} APIs | Eval: {len(evals)} examples")

## Build FAISS Index and Embed Queries

In [ ]:
from tqdm import tqdm

index = build_faiss_index(corpus_matrix)
print(f"FAISS index ready: {index.ntotal} vectors")

queries = [e["user_query"] for e in evals]
query_embeddings = get_embeddings(queries)
print(f"Query embeddings: {query_embeddings.shape}")

## Retrieve and Evaluate

For each query, retrieve the top-10 APIs from the full corpus and compute Recall@1, Recall@5, Recall@10, and MRR.

In [ ]:
K = 10
all_retrieved = []
all_gt_indices = []

for i, ex in enumerate(evals):
    top_k_indices = retrieve_top_k(query_embeddings[i], index, k=K)
    all_retrieved.append(top_k_indices)
    gt_indices = [name_to_idx[name] for name in ex["ground_truth_apis"] if name in name_to_idx]
    all_gt_indices.append(gt_indices)

results = evaluate_batch(all_retrieved, all_gt_indices, ks=[1, 5, 10])
print("\n=== Baseline Results (Full Corpus) ===")
for metric, score in results.items():
    print(f"  {metric}: {score:.4f}")

## Save Results

In [ ]:
import json
with open(PROJECT_DIR / 'results_baseline.json', 'w') as f:
    json.dump({'condition': 'random_full_corpus', 'results': results}, f, indent=2)
print("Saved results_baseline.json")